# Helper Functions
> These are for helping any other functions in other files

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
def send_to_schema(df, table_name, schema):
    df.write.mode("overwrite").saveAsTable(f"formone.{schema}.{table_name}")
    print(f"✅ Successfully sent {table_name} to {schema}")

In [0]:
def convert_time_to_ms(col_name):
    col = F.col(col_name)
    
    # For format "mm:ss.SSS"
    has_colon = col.contains(":")
    
    minutes = F.split(col, ":")[0].cast(IntegerType())
    sec_parts_with_colon = F.split(F.split(col, ":")[1], "\\.")
    seconds_with_colon = sec_parts_with_colon[0].cast(IntegerType())
    ms_with_colon = F.coalesce(sec_parts_with_colon[1].cast(IntegerType()), F.lit(0))
    result_with_colon = (minutes * 60 * 1000) + (seconds_with_colon * 1000) + ms_with_colon

    # For format "ss.SSS"
    sec_parts_no_colon = F.split(col, "\\.")
    seconds_no_colon = sec_parts_no_colon[0].cast(IntegerType())
    ms_no_colon = F.coalesce(sec_parts_no_colon[1].cast(IntegerType()), F.lit(0))
    result_no_colon = (seconds_no_colon * 1000) + ms_no_colon

    return F.when(col.isNull(), F.lit(None).cast(IntegerType())) \
            .when(has_colon, result_with_colon) \
            .otherwise(result_no_colon)

In [0]:
def clean_wall_clock(col_name):
    return F.regexp_replace(F.col(col_name), r"(\d{1,2}:\d{2}:\d{2}).*", "$1")